In [2]:
# Objective  - Load the orders data and create an RDD
# Apply a bunch of transformations

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("MySparkApp") \
    .getOrCreate()

In [3]:
spark

In [4]:
from google.colab import files
uploaded = files.upload()

Saving orders.txt to orders.txt


In [6]:
#Count the orders under each status
orders_rdd = spark.sparkContext.textFile("orders.txt")

In [7]:
orders_rdd.take(5)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE']

In [8]:
mapped_rdd = orders_rdd.map(lambda x: (x.split(",")[3],1))

In [9]:
mapped_rdd.take(5)

[('CLOSED', 1),
 ('PENDING_PAYMENT', 1),
 ('COMPLETE', 1),
 ('CLOSED', 1),
 ('COMPLETE', 1)]

In [11]:
reduced_rdd = mapped_rdd.reduceByKey(lambda x,y: x+y)

In [12]:
reduced_rdd.collect()

[('CLOSED', 7556),
 ('COMPLETE', 22899),
 ('PROCESSING', 8275),
 ('PAYMENT_REVIEW', 729),
 ('CANCELED', 1428),
 ('SUSPECTED_FRAUD', 1558),
 ('PENDING_PAYMENT', 15030),
 ('PENDING', 7610),
 ('ON_HOLD', 3798)]

In [13]:
reduced_sorted  = reduced_rdd.sortBy(lambda x:x[1])

In [14]:
reduced_sorted.collect()

[('PAYMENT_REVIEW', 729),
 ('CANCELED', 1428),
 ('SUSPECTED_FRAUD', 1558),
 ('ON_HOLD', 3798),
 ('CLOSED', 7556),
 ('PENDING', 7610),
 ('PROCESSING', 8275),
 ('PENDING_PAYMENT', 15030),
 ('COMPLETE', 22899)]

In [16]:
# In descending order
reduced_rdd.sortBy(lambda x:x[1],False).collect()

[('COMPLETE', 22899),
 ('PENDING_PAYMENT', 15030),
 ('PROCESSING', 8275),
 ('PENDING', 7610),
 ('CLOSED', 7556),
 ('ON_HOLD', 3798),
 ('SUSPECTED_FRAUD', 1558),
 ('CANCELED', 1428),
 ('PAYMENT_REVIEW', 729)]

In [18]:
# Find the premium customers (Top 10 who placed the most number of orders)
customers_mapped = orders_rdd.map(lambda x: (x.split(",")[2],1))
customers_mapped.take(5)

[('11599', 1), ('256', 1), ('12111', 1), ('8827', 1), ('11318', 1)]

In [19]:
customers_agg = customers_mapped.reduceByKey(lambda x,y:x+y)

In [20]:
customers_agg.take(20)

[('11599', 5),
 ('256', 10),
 ('12111', 6),
 ('8827', 6),
 ('7130', 7),
 ('4530', 10),
 ('2911', 6),
 ('918', 5),
 ('9488', 7),
 ('333', 6),
 ('4367', 5),
 ('9503', 3),
 ('10039', 10),
 ('6983', 6),
 ('5793', 3),
 ('4840', 2),
 ('5649', 2),
 ('11586', 9),
 ('8214', 5),
 ('8136', 7)]

In [23]:
customers_sorted = customers_agg.sortBy(lambda x: x[1],False)

In [24]:
customers_sorted.take(10)

[('6316', 16),
 ('12431', 16),
 ('5897', 16),
 ('569', 16),
 ('221', 15),
 ('4320', 15),
 ('5283', 15),
 ('12284', 15),
 ('5654', 15),
 ('5624', 15)]

In [26]:
#Distinct count of customers who placed at least one order
distinct_customers = orders_rdd.map(lambda x: x.split(",")[2]).distinct()

In [28]:
distinct_customers.count()

12405

In [29]:
orders_rdd.count()

68883

In [30]:
#Which customers has the max number of CLOSED orders
filtered_orders = orders_rdd.filter(lambda x: (x.split(",")[3]=="CLOSED"))

In [31]:
filtered_orders.take(20)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '12,2013-07-25 00:00:00.0,1837,CLOSED',
 '18,2013-07-25 00:00:00.0,1205,CLOSED',
 '24,2013-07-25 00:00:00.0,11441,CLOSED',
 '25,2013-07-25 00:00:00.0,9503,CLOSED',
 '37,2013-07-25 00:00:00.0,5863,CLOSED',
 '51,2013-07-25 00:00:00.0,12271,CLOSED',
 '57,2013-07-25 00:00:00.0,7073,CLOSED',
 '61,2013-07-25 00:00:00.0,4791,CLOSED',
 '62,2013-07-25 00:00:00.0,9111,CLOSED',
 '87,2013-07-25 00:00:00.0,3065,CLOSED',
 '90,2013-07-25 00:00:00.0,9131,CLOSED',
 '101,2013-07-25 00:00:00.0,5116,CLOSED',
 '116,2013-07-26 00:00:00.0,8763,CLOSED',
 '129,2013-07-26 00:00:00.0,9937,CLOSED',
 '133,2013-07-26 00:00:00.0,10604,CLOSED',
 '191,2013-07-26 00:00:00.0,16,CLOSED',
 '201,2013-07-26 00:00:00.0,9055,CLOSED',
 '211,2013-07-26 00:00:00.0,10372,CLOSED']

In [32]:
filtered_mapped = filtered_orders.map(lambda x: (x.split(",")[2],1))

In [33]:
filtered_mapped.take(20)

[('11599', 1),
 ('8827', 1),
 ('1837', 1),
 ('1205', 1),
 ('11441', 1),
 ('9503', 1),
 ('5863', 1),
 ('12271', 1),
 ('7073', 1),
 ('4791', 1),
 ('9111', 1),
 ('3065', 1),
 ('9131', 1),
 ('5116', 1),
 ('8763', 1),
 ('9937', 1),
 ('10604', 1),
 ('16', 1),
 ('9055', 1),
 ('10372', 1)]

In [38]:
filtered_mapped.reduceByKey(lambda x,y: x+y).sortBy(lambda x:x[1],False).take(5)

[('1833', 6), ('1363', 5), ('1687', 5), ('5493', 5), ('5011', 4)]